In [1]:
# !pip install quantizeml

In [2]:
# !pip install cnn2snn==2.17.0


In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# from google.colab import drive
# drive.mount('/content/gdrive')

# !ls gdrive/MyDrive/


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import copy
import numpy as np

def to_one_hot(y, num_classes):
    return y if y.ndim == 2 else torch.nn.functional.one_hot(y, num_classes=num_classes).float()

class AsymmetricPaddedConv(nn.Module):
    def __init__(self, original_conv, pads_list):
        super().__init__()
        self.pads_list = pads_list
        self.conv = original_conv
        self.conv.padding = (0, 0)

    def forward(self, x):
        x = F.pad(x, self.pads_list, "constant", 0)
        return self.conv(x)

class SplittedDWConv(nn.Module):
    def __init__(self, original_conv):
        super().__init__()
        self.dw_k5_s1 = copy.deepcopy(original_conv)
        self.dw_k5_s1.stride = (1, 1)
        self.dw_k5_s1.padding = (2, 2)
        num_channels = original_conv.in_channels

        identity_conv = nn.Conv2d(
            in_channels=num_channels,
            out_channels=num_channels,
            kernel_size=3,
            stride=2,
            padding=0,
            groups=num_channels,
            bias=False
        )

        with torch.no_grad():
            identity_weights = torch.zeros_like(identity_conv.weight)
            identity_weights[:, 0, 1, 1] = 1
            identity_conv.weight.copy_(identity_weights)
            identity_conv.weight.requires_grad = False

        self.dw_k3_s2_identity = AsymmetricPaddedConv(
            identity_conv,
            pads_list=[0, 1, 0, 1]
        )

    def forward(self, x):
        x = self.dw_k5_s1(x)
        x = self.dw_k3_s2_identity(x)
        return x

def get_module_by_name(model, name):
    names = name.split('.')
    module = model
    for n in names:
        module = getattr(module, n)
    return module

def set_module_by_name(model, name, new_module):
    names = name.split('.')
    parent_module = model
    for i in range(len(names) - 1):
        parent_module = getattr(parent_module, names[i])
    setattr(parent_module, names[-1], new_module)


def patch_mobilenetv4_for_akida(model_to_patch):
    model = copy.deepcopy(model_to_patch)
    padding_map = {
        'conv_stem': [0, 1, 0, 1],
        'blocks.0.0.conv': [0, 1, 0, 1],
        'blocks.1.0.conv': [0, 1, 0, 1],
        'blocks.3.0.dw_mid.conv': [0, 1, 0, 1],
        'blocks.2.0.dw_mid.conv': [1, 2, 1, 2],
    }

    print("Starting model patching process")

    module_names = [name for name, _ in model.named_modules()]

    for name in module_names:
        try:
            module = get_module_by_name(model, name)
        except AttributeError:
            continue
        if isinstance(module, nn.Conv2d) and module.kernel_size == (5, 5) and module.stride == (2, 2) and module.groups == module.in_channels:
            print(f"  - Found unsupported DWConv. Replacing '{name}'...")
            unsupported_layer = get_module_by_name(model, name)
            new_layer = SplittedDWConv(unsupported_layer)
            set_module_by_name(model, name, new_layer)
            continue
        if name in padding_map:
            print(f"  - Found layer for padding fix. Replacing '{name}'...")
            original_layer = get_module_by_name(model, name)
            pads_list = padding_map[name]
            new_layer = AsymmetricPaddedConv(original_layer, pads_list)
            set_module_by_name(model, name, new_layer)

    print("Patching complete.")
    return model

# Example usage:
# original_model = timm.create_model(
#     "mobilenetv4_conv_small.e2400_r224_in1k",
#     pretrained=True,
#     num_classes= X #enter your number of classes here
# ).eval()
# patched_model = patch_mobilenetv4_for_akida(original_model)

#this patched model is compatible with Akida.

/home/akayyam/miniconda3/envs/torch_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## model

In [6]:

import torch, torch.nn as nn, torch.nn.functional as F
import timm


class CFG:
    LEFT_IMG_DIR = "../content/leftImg8bit"
    GT_FINE_DIR  = "../content/gtFine"
    TILE_SIZE    = 384
    BATCH_SIZE   = 4
    EPOCHS       = 100
    LR           = 5e-4
    WEIGHT_DECAY = 1e-4
    NUM_CLASSES  = 19
    IGNORE_INDEX = 255



# -------------------- Akida-friendly Decoder --------------------
class SepConvBNReLU(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.dw = nn.Conv2d(in_c,in_c,3,1,1,groups=in_c,bias=False)
        self.bn1= nn.BatchNorm2d(in_c)
        self.pw = nn.Conv2d(in_c,out_c,1,1,0,bias=False)
        self.bn2= nn.BatchNorm2d(out_c)
        self.act= nn.ReLU(inplace=False)
    def forward(self,x):
        x=self.act(self.bn1(self.dw(x)))
        x=self.act(self.bn2(self.pw(x)))
        return x

class UpBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.up=nn.ConvTranspose2d(in_c,out_c,4,2,1,bias=False)
        self.bn=nn.BatchNorm2d(out_c)
        self.act=nn.ReLU(inplace=False)
    def forward(self,x): return self.act(self.bn(self.up(x)))

class DecoderStage(nn.Module):
    def __init__(self,in_c,skip_c,out_c):
        super().__init__()
        self.up=UpBlock(in_c,out_c)
        self.c1=SepConvBNReLU(out_c+skip_c,out_c)
        self.c2=SepConvBNReLU(out_c,out_c)
    def forward(self,x,skip):
        x=self.up(x)
        if x.shape[-2:]!=skip.shape[-2:]:
            x=F.pad(x,(0,skip.shape[-1]-x.shape[-1],0,skip.shape[-2]-x.shape[-2]))
        x=torch.cat([x,skip],1)
        x=self.c1(x); x=self.c2(x)
        return x

class UNetV4_AkidaDecoder(nn.Module):
    def __init__(self,num_classes=19):
        super().__init__()
        self.encoder=timm.create_model("mobilenetv4_conv_small.e1200_r224_in1k",
                                       pretrained=True,features_only=True)

        self.encoder = patch_mobilenetv4_for_akida(self.encoder)

        chs=self.encoder.feature_info.channels()
        self.stage_idx=[-1,-2,-3,-4,-5][:len(chs)]
        enc_ch=[chs[i] for i in self.stage_idx]
        bottleneck_c=enc_ch[0]
        dec_plan=[256,128,96,64,48][:len(enc_ch)-1]
        self.stages=nn.ModuleList()
        in_c=bottleneck_c
        for out_c,skip_c in zip(dec_plan,enc_ch[1:]):
            self.stages.append(DecoderStage(in_c,skip_c,out_c)); in_c=out_c
        # self.head=nn.Conv2d(in_c,num_classes,1)
        self.head = nn.ConvTranspose2d(
            in_c, num_classes,
            kernel_size=2, stride=2
        )

    def forward(self,x):
        feats=self.encoder(x)
        ordered=[feats[i] for i in self.stage_idx]
        y=ordered[0]
        for st,sk in zip(self.stages,ordered[1:]): y=st(y,sk)
        y=self.head(y)
        # if y.shape[-2:]!=x.shape[-2:]:
        #     y=F.interpolate(y,size=x.shape[-2:],mode="nearest")
        return y

# -------------------- Build model & load your trained weights --------------------
model = UNetV4_AkidaDecoder(num_classes=CFG.NUM_CLASSES).to(device)
# ckpt_path = "gdrive/MyDrive/Akida/Akida_seg_tile.pth"  # your saved file
ckpt_path = "../checkpoints/Akida_seg_tile.pth"  # your saved file

state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state)
model.eval()
print(" Model loaded:", ckpt_path)

Unexpected keys (norm_head.num_batches_tracked, classifier.bias, classifier.weight, conv_head.weight, norm_head.bias, norm_head.running_mean, norm_head.running_var, norm_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Starting model patching process
  - Found layer for padding fix. Replacing 'conv_stem'...
  - Found layer for padding fix. Replacing 'blocks.0.0.conv'...
  - Found layer for padding fix. Replacing 'blocks.1.0.conv'...
  - Found unsupported DWConv. Replacing 'blocks.2.0.dw_mid.conv'...
  - Found layer for padding fix. Replacing 'blocks.3.0.dw_mid.conv'...
Patching complete.
 Model loaded: ../checkpoints/Akida_seg_tile.pth


In [7]:
# !pip install onnx
# !pip install onnxruntime

In [8]:
torch.__version__

'2.7.0+cu126'

In [9]:

# from quantizeml.models import QuantizationParams, quantize
# import torch.onnx

# dummy_input = torch.randn(1, 3, 384, 384, dtype=torch.float32).to(device)
# torch.onnx.export(
#     model,
#     dummy_input,
#     "gdrive/MyDrive/Akida/Akida_seg_tile.onnx",
#     input_names=["input"],
#     output_names=["output"],
#     opset_version=12,
#     dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
#     dynamo=False  ## THIS WAS CRITICAL TO RUN THIS SAFELY!!! FUCK!!!!
# )


In [10]:
# import onnx
# onnx_model = onnx.load("gdrive/MyDrive/Akida/Akida_seg_tile.onnx")
# onnx.checker.check_model(onnx_model)
# print("ONNX model is valid!")

In [11]:
# import onnxruntime as ort
# import numpy as np

# session = ort.InferenceSession("gdrive/MyDrive/Akida/Akida_seg_tile.onnx")
# input_name = session.get_inputs()[0].name

# x_test = torch.randn(1, 3, 384, 384, dtype=torch.float32)
# outputs = session.run(None, {input_name: x_test.numpy()})
# print("Output shape:", outputs[0].shape)

In [12]:
import quantizeml
print(quantizeml.__version__)

2025-12-15 19:02:05.767150: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765854126.173509 1490676 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765854126.294929 1490676 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765854127.220585 1490676 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765854127.220613 1490676 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765854127.220616 1490676 computation_placer.cc:177] computation placer alr

1.0.1


In [13]:
# import onnx
# onnx_model = onnx.load_model("gdrive/MyDrive/Akida/Akida_seg_tile.onnx")

# ops = set([n.op_type for n in onnx_model.graph.node]); print(ops)

# x_test = torch.randn(4, 3, 384, 384, dtype=torch.float32)
# x_test_np = x_test.numpy()

# model_quantized = quantize(onnx_model, samples=x_test_np)

In [ ]:

import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

import onnx
from quantizeml.models import quantize

# ----------------------------
# Config
# ----------------------------
TILE_SIZE = 384
STRIDE = TILE_SIZE // 2      # 50% overlap -> 192
IMG_DIR = "../content/leftImg8bit/train/aachen/"
ONNX_PATH = "../checkpoints/Akida_seg_tile.onnx"
MAX_SAMPLES = 5000            # cap representative samples to control memory/time

# ----------------------------
# Albumentations transform
# NOTE: No Resize; we keep original resolution in patches.
# ----------------------------
val_tf_norm = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()  # -> torch tensor (C,H,W) float32
])

# ----------------------------
# Utilities
# ----------------------------
def pad_to_multiple(image, multiple=384, pad_value=(0, 0, 0)):
    """
    Pad HWC RGB image so H and W become multiples of 'multiple'.
    Returns padded image and (top, bottom, left, right) pad amounts.
    """
    h, w = image.shape[:2]
    pad_h = (multiple - (h % multiple)) % multiple
    pad_w = (multiple - (w % multiple)) % multiple
    if pad_h == 0 and pad_w == 0:
        return image, (0, 0, 0, 0)

    top, bottom, left, right = 0, pad_h, 0, pad_w
    padded = cv2.copyMakeBorder(
        image, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=pad_value
    )
    return padded, (top, bottom, left, right)


def extract_patches(image, tile_size=384, stride=192):
    """
    Extract overlapping HWC (RGB) patches of size tile_size x tile_size with given stride.
    Image is padded so patches tile the canvas.
    Returns: list of patches (HWC) and a list of (y, x) top-left coords on the padded image.
    """
    padded, _ = pad_to_multiple(image, multiple=tile_size, pad_value=(0, 0, 0))
    H, W = padded.shape[:2]

    patches = []
    coords = []
    for y in range(0, H - tile_size + 1, stride):
        for x in range(0, W - tile_size + 1, stride):
            patch = padded[y:y + tile_size, x:x + tile_size]
            patches.append(patch)
            coords.append((y, x))
    return patches, coords


# ----------------------------
# Collect representative patches
# ----------------------------
img_paths = [os.path.join(IMG_DIR, f) for f in os.listdir(IMG_DIR) if f.lower().endswith(".png")]

samples = []
patch_index_meta = []  # (img_path, y, x) mapping for optional stitching later

for img_path in img_paths:
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue

    # Ensure 3-channel RGB
    if img_bgr.ndim == 2:
        img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
    elif img_bgr.shape[2] == 4:
        img_bgr = img_bgr[:, :, :3]  # drop alpha

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Extract overlapping patches
    patches, coords = extract_patches(img_rgb, tile_size=TILE_SIZE, stride=STRIDE)

    for (patch, (y, x)) in zip(patches, coords):
        aug = val_tf_norm(image=patch)
        tensor_chw = aug["image"]  # torch.Tensor (C, 384, 384), float32

        samples.append(tensor_chw.numpy())        # (C, 384, 384) float32
        patch_index_meta.append((img_path, y, x))

        if len(samples) >= MAX_SAMPLES:
            break
    if len(samples) >= MAX_SAMPLES:
        break

if len(samples) == 0:
    raise RuntimeError("No patches were collected. Check IMG_DIR or image formats.")

# Stack into a batch: (N, C, 384, 384)
samples_np = np.stack(samples, axis=0).astype(np.float32)

# ----------------------------
# Load and quantize ONNX model
# ----------------------------
onnx_model = onnx.load_model(ONNX_PATH)

model_quantized = quantize(
    onnx_model,
    samples=samples_np,      # representative dataset from overlapping patches
    # skip_optimization=True  # uncomment if your quantizer requires it
)



Applied 4 of general pattern rewrite rules.
Applied 5 of general pattern rewrite rules.


In [ ]:
from cnn2snn import convert

model_akida = convert(model_quantized)
model_akida.summary()

model_akida.save("gdrive/MyDrive/Akida/Akida_seg_tile_10K.h5")

                  Model Summary                   
__________________________________________________
Input shape    Output shape    Sequences  Layers
[384, 384, 3]  [384, 384, 19]  1          83    
__________________________________________________

__________________________________________________________________________________________________________________________
Layer (type)                                                                            Output shape    Kernel shape    

================================= SW//encoder/conv_stem/conv/Conv-dequantizer (Software) =================================

/encoder/conv_stem/conv/Conv (InputConv2D)                                              [192, 192, 32]  (3, 3, 3, 32)   
__________________________________________________________________________________________________________________________
/encoder/act1/Relu_output_0_identity_conv (Conv2D)                                      [192, 192, 32]  (1, 1, 32, 32)  
______________